In [278]:
import collections
import json
import pickle
import re

import pandas as pd

import importlib
import arkham_reload
importlib.reload(arkham_reload)
arkham_reload.reload_arkham_modules()


# Configuration
TABOO_FLAG = 'CURRENT'  # match combined.ipynb behaviour

DECKLIST_JSON_PICKLE = 'decklist_json.pickle'
CARD_JSON_PICKLE = 'card_json.pickle'
TABOO_FILE = 'taboo.json'

# Load previously-scraped decklists and cards from disk
with open(DECKLIST_JSON_PICKLE, 'rb') as f:
  decklist_json = pickle.loads(f.read())
with open(CARD_JSON_PICKLE, 'rb') as f:
  card_json = pickle.loads(f.read())

print(f"Loaded {len(decklist_json)} raw decklists")
print(f"Loaded {len(card_json)} cards")

Loaded 63270 raw decklists
Loaded 2046 cards


In [279]:
# Basic decklist cleaning: drop empty entries, joke decks, extreme copy violations

from arkham_popularity import clean_decklist_json

decklist_json, removed_decks = clean_decklist_json(
  decklist_json,
  card_json,
  TABOO_FILE,
)

for deck_id, reason, detail in removed_decks:
  label = detail or ''
  print(f"  removed {deck_id} ({reason}): {label}")

print(f"{len(decklist_json)} decklists after removing empty/joke/illegal copies")

  removed 25318 (empty): 
  removed 25293 (empty): 
  removed 25292 (empty): 
  removed 25291 (empty): 
  removed 25245 (empty): 
  removed 25243 (empty): 
  removed 25241 (empty): 
  removed 25240 (empty): 
  removed 25239 (empty): 
  removed 25237 (empty): 
  removed 25236 (empty): 
  removed 25235 (empty): 
  removed 25234 (empty): 
  removed 25232 (empty): 
  removed 25224 (empty): 
  removed 25223 (empty): 
  removed 25222 (empty): 
  removed 25221 (empty): 
  removed 25206 (empty): 
  removed 25204 (empty): 
  removed 25200 (empty): 
  removed 25199 (empty): 
  removed 25195 (empty): 
  removed 25194 (empty): 
  removed 25193 (empty): 
  removed 25190 (empty): 
  removed 25189 (empty): 
  removed 25188 (empty): 
  removed 25187 (empty): 
  removed 25179 (empty): 
  removed 25176 (empty): 
  removed 25170 (empty): 
  removed 25168 (empty): 
  removed 25129 (empty): 
  removed 25128 (empty): 
  removed 25115 (empty): 
  removed 25112 (empty): 
  removed 25111 (empty): 
  removed 25

In [280]:
# Load taboo data and helpers (adapted from combined.ipynb)

with open(TABOO_FILE, 'r', encoding='utf-8') as f:
  taboo_json = json.load(f)

# Find id of most recent taboo
MAX_TABOO = max([taboo_dict['id'] for taboo_dict in taboo_json])
print(f"Most recent taboo id: {MAX_TABOO}")

# Build lookup: taboo_id -> {card_code -> taboo_card}
taboo_dict = collections.defaultdict(dict)
for taboo in taboo_json:
  taboo_id = taboo['id']
  for taboo_card in json.loads(taboo['cards']):
    card_code = taboo_card['code']
    taboo_dict[taboo_id][card_code] = taboo_card

  if taboo_id == MAX_TABOO:
    # list of card_codes in most recent taboo that change xp or are forbidden
    tabooed_cards = [card['code'] for card in json.loads(taboo['cards'])
                     if 'xp' in card or card['text'] == 'Forbidden.']


def lookup_taboo(card_code: str, taboo_id: int | None):
  """Return taboo card entry for (card_code, taboo_id), or None if missing."""
  if taboo_id in taboo_dict:
    return taboo_dict[taboo_id].get(card_code, None)
  return None


def lookup_taboo_xp(card_code: str, taboo_id: int | None) -> int:
  """Return taboo xp modifier for (card_code, taboo_id), defaulting to 0."""
  taboo_card = lookup_taboo(card_code, taboo_id)
  if taboo_card is not None:
    return taboo_card.get('xp', 0)
  return 0

Most recent taboo id: 10


In [281]:
# Map card_id -> canonical_id (spec.md fingerprint + duplicate_of_code)
# and rewrite decklist slots / taboo keys to canonical_id.

from arkham_canonical import CanonicalMapper, parse_investigator_front_back

canonical_mapper = CanonicalMapper(card_json, chapter=1)
canonical_id_map = canonical_mapper.canonical_id_map
canonical_cycle = canonical_mapper.canonical_cycle


def to_canonical(card_id: str) -> str:
  """Return canonical_id for a card_id (identity if unmapped)."""
  return canonical_mapper.to_canonical(card_id)


def to_canonical_front(card_id: str) -> str:
  """Return canonical_front for an investigator front card_id."""
  return canonical_mapper.to_canonical_front(card_id)


def to_canonical_back(card_id: str) -> str:
  """Return canonical_back for an investigator back card_id."""
  return canonical_mapper.to_canonical_back(card_id)


def decklist_canonical_front_back(decklist: dict) -> tuple[str, str]:
  """Return (canonical_front, canonical_back) for a decklist."""
  return canonical_mapper.decklist_canonical_front_back(decklist)


def cycle_for_slot(card_id: str) -> int | None:
  """Return CanonicalCard.cycle for a slot code, or None if out-of-order or missing."""
  return canonical_mapper.cycle_for_slot(card_id)


def decklist_cycle(slots: dict) -> int | None:
  """Return Decklist.cycle: max cycle among known slot codes."""
  return canonical_mapper.decklist_cycle(slots)


def decklist_has_unknown_slots(slots: dict) -> bool:
  """Return True if any slot code is missing from card_json."""
  return canonical_mapper.decklist_has_unknown_slots(slots)


# Merge decklist card codes
for decklist in decklist_json.values():
  slots = decklist['slots']
  for card_code in list(slots.keys()):
    canonical_id = to_canonical(card_code)
    if canonical_id == card_code:
      continue

    num = slots.pop(card_code)
    slots[canonical_id] = slots.get(canonical_id, 0) + num


# Merge taboo card codes so taboo lookups stay consistent
for taboo_id in sorted(taboo_dict.keys()):
  taboo = taboo_dict[taboo_id]
  for card_code, taboo_card in list(taboo.items()):
    canonical_id = to_canonical(card_code)
    if canonical_id == card_code:
      continue
    taboo_dict[taboo_id][canonical_id] = taboo_card

# tabooed_cards is used for CURRENT-taboo xp checks on canonical slot codes
tabooed_cards = sorted({to_canonical(card_code) for card_code in tabooed_cards})

reprint_card_ids = sum(1 for card_id, canonical_id in canonical_id_map.items()
                       if card_id != canonical_id)
print(f"Mapped {len(card_json)} card_ids -> {len(set(canonical_id_map.values()))} canonical_ids")
print(f"Merged {reprint_card_ids} reprint card_ids in decklists and taboo data")

Mapped 2046 card_ids -> 1653 canonical_ids
Merged 143 reprint card_ids in decklists and taboo data


In [282]:
# Cycle stats (spec Pack Order; uses CanonicalCard.cycle from arkham_canonical)

cycle_card_count = collections.Counter()
for card_code, card in card_json.items():
  # Ignore investigator-specific signature cards
  if 'restrictions' in card and 'investigator' in card['restrictions']:
    continue
  # Ignore basic weaknesses
  if card.get('subtype_code') == 'basicweakness':
    continue
  # Count each canonical_id once
  if to_canonical(card_code) != card_code:
    continue

  cycle = cycle_for_slot(card_code)
  if cycle is None:
    continue
  cycle_card_count[cycle] += 1

cycle_card_cumulative: dict[int, int] = {}
so_far = 0
for cycle in sorted(cycle_card_count):
  so_far += cycle_card_count[cycle]
  cycle_card_cumulative[cycle] = so_far

deck_cycle_by_card_cycle: dict[int, dict[int, int]] = collections.defaultdict(
  lambda: collections.defaultdict(int)
)
unknown_slot_refs = 0
for decklist in decklist_json.values():
  slots = decklist.get('slots') or {}
  deck_cycle = decklist_cycle(slots)
  if deck_cycle is None:
    continue

  for card_code, num in slots.items():
    if not canonical_mapper.is_known_card(card_code):
      unknown_slot_refs += num
      continue
    card_cycle = cycle_for_slot(card_code)
    if card_cycle is None:
      continue
    deck_cycle_by_card_cycle[deck_cycle][card_cycle] += num

cycle_pivot = pd.DataFrame.from_dict(
  {deck_cycle: dict(card_counts) for deck_cycle, card_counts in deck_cycle_by_card_cycle.items()},
  orient='index',
).fillna(0).astype(int)
cycle_pivot.index.name = 'Decklist.cycle'
cycle_pivot.columns.name = 'CanonicalCard.cycle'
cycle_pivot = cycle_pivot.sort_index().reindex(sorted(cycle_pivot.columns), axis=1)

cycle_pivot['All'] = cycle_pivot.sum(axis=1)
all_row = cycle_pivot.drop(columns='All').sum(axis=0)
all_row['All'] = int(cycle_pivot.drop(columns='All').values.sum())
cycle_pivot.loc['All'] = all_row

print("Cycle card counts:")
print(dict(sorted(cycle_card_count.items())))
print("Cumulative cards by cycle:")
print(cycle_card_cumulative)
print("Slot copies by Decklist.cycle x CanonicalCard.cycle (margins reproduce prior totals):")
display(cycle_pivot)
if unknown_slot_refs:
  print(f"Skipped {unknown_slot_refs} slot copies with unknown card codes")

Cycle card counts:
{1: 86, 2: 107, 3: 112, 4: 114, 5: 105, 6: 96, 7: 122, 8: 116, 9: 139, 10: 114, 11: 119, 12: 126, 13: 215}
Cumulative cards by cycle:
{1: 86, 2: 193, 3: 305, 4: 419, 5: 524, 6: 620, 7: 742, 8: 858, 9: 997, 10: 1111, 11: 1230, 12: 1356, 13: 1571}
Slot copies by Decklist.cycle x CanonicalCard.cycle (margins reproduce prior totals):


CanonicalCard.cycle,1,2,3,4,5,6,7,8,9,10,11,12,13,All
Decklist.cycle,,,,,,,,,,,,,,
1,74870,0,0,0,0,0,0,0,0,0,0,0,0,74870
2,141653,57272,0,0,0,0,0,0,0,0,0,0,0,198925
3,103231,41415,35143,0,0,0,0,0,0,0,0,0,0,179789
4,80831,35052,33287,27512,0,0,0,0,0,0,0,0,0,176682
5,99197,41843,37745,31337,44610,0,0,0,0,0,0,0,0,254732
6,50737,23066,22416,20074,24568,19679,0,0,0,0,0,0,0,160540
7,65499,24976,17981,16459,14573,9331,42663,0,0,0,0,0,0,191482
8,33910,14433,11861,10985,13345,10963,12111,16776,0,0,0,0,0,124384
9,61894,21765,21064,15316,15844,12034,18459,9164,27043,0,0,0,0,202583


Skipped 1 slot copies with unknown card codes


In [283]:
# Normalize by row
cycle_pivot.div(cycle_pivot.sum(axis=1) / 2, axis=0)

CanonicalCard.cycle,1,2,3,4,5,6,7,8,9,10,11,12,13,All
Decklist.cycle,,,,,,,,,,,,,,
1,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,0.712092,0.287908,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
3,0.574179,0.230353,0.195468,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
4,0.457494,0.198390,0.188401,0.155715,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
5,0.389417,0.164263,0.148175,0.123019,0.175125,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
6,0.316040,0.143678,0.139629,0.125040,0.153034,0.122580,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
7,0.342063,0.130435,0.093904,0.085956,0.076106,0.048730,0.222804,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
8,0.272623,0.116036,0.095358,0.088315,0.107289,0.088138,0.097368,0.134873,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
9,0.305524,0.107437,0.103977,0.075604,0.078210,0.059403,0.091118,0.045236,0.133491,0.000000,0.000000,0.000000,0.000000,1.0


In [284]:
# XP and investigator name helpers (adapted from combined.ipynb)

UPGRADE_PATTERN = re.compile(r"\d+\|\d+.*")


def parse_customizable(customizable_string: str):
  """Parse ArkhamDB customizable upgrade string.

  Returns (indices_list, xp_list) where each index is an upgrade index and
  each xp is its cost, filtered to upgrades with positive xp.
  """
  upgrade_list = customizable_string.split(',')
  upgrade_list = [u.split('|') for u in upgrade_list
                  if UPGRADE_PATTERN.match(u)]
  indices_list = [u[0] for u in upgrade_list if int(u[1]) > 0]
  xp_list = [int(u[1]) for u in upgrade_list if int(u[1]) > 0]
  return indices_list, xp_list


def calc_card_code_xp(card_code: str, taboo_id: int | None) -> int:
  card_dict = card_json[card_code]
  card_xp = card_dict.get('xp', 0)
  exceptional = card_dict.get('exceptional', False)

  # Apply taboo modifications if we are pinning to the latest taboo
  if TABOO_FLAG == 'CURRENT' and card_code in tabooed_cards:
    card_taboo = lookup_taboo(card_code, MAX_TABOO)
    if card_taboo is not None:
      if 'xp' in card_taboo:
        card_xp += card_taboo['xp']
      if 'exceptional' in card_taboo:
        exceptional = card_taboo['exceptional']

  if exceptional:
    card_xp *= 2

  return card_xp


def decklist_investigator_name(decklist: dict) -> str:
  """Return investigator name, treating parallel fronts/backs as distinct."""
  investigator_name = decklist['investigator_name']
  return investigator_name


def decklist_investigator_front_back(decklist: dict) -> tuple[str, str]:
  """Return raw investigator_front/back card_ids from a decklist."""
  return parse_investigator_front_back(decklist)


def decklist_xp(decklist: dict) -> int:
  """Return total XP of the decklist, including customizable upgrades."""
  xp = 0
  taboo_id = decklist.get('taboo_id')

  for card_code, num in decklist['slots'].items():
    card_dict = card_json[card_code]
    card_xp = calc_card_code_xp(card_code, taboo_id)

    if card_dict.setdefault('myriad', False):
      xp += card_xp
    else:
      xp += card_xp * num

  meta = decklist.get('meta', '')
  if meta:
    customizable_cards = json.loads(meta)
    for key, customizable_string in customizable_cards.items():
      if key.startswith('cus_'):
        _, xp_list = parse_customizable(customizable_string)
        xp += sum(xp_list)

  # Global XP adjustments
  if '08125' in decklist['slots']:
    xp = max(0, xp - 3)  # In the Thick of It

  if decklist_investigator_name(decklist) == 'Kymani Jones':
    xp = max(0, xp - 5)

  return xp

In [285]:
# Build decklist DataFrame via spec.md pipeline (D1–D4)
# Tolerates missing card_json entries until cards are re-scraped.
# Re-run this cell after editing arkham_*.py (reloads modules + rebuilds pop_engine).

from arkham_popularity import ArkhamPopularityEngine, prepared_decks_to_dataframe

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)
prepared_decks = pop_engine.prepare_all(decklist_json)
df_all_decks = prepared_decks_to_dataframe(prepared_decks)

print(df_all_decks.shape)
print(
  'Unknown slots:',
  int(df_all_decks['has_unknown_slots'].sum()),
  '| Chapter 2:',
  int(df_all_decks['has_chapter_2_cards'].sum()),
)
print(df_all_decks[[
  'id', 'investigator_name',
  'canonical_front', 'canonical_back',
  'cycle', 'xp_cost', 'is_ignore',
]].head())

(55809, 21)
Unknown slots: 1 | Chapter 2: 539
      id investigator_name canonical_front canonical_back  cycle  xp_cost  \
0  25345       Agnes Baker           01004          01004    5.0        0   
1  25344       Wendy Adams           01005          01005    4.0        0   
2  25343      Stella Clark           60501          60501    7.0        0   
3  25342       Agnes Baker           01004          01004    2.0        0   
4  25341      Roland Banks           01001          01001    1.0        5   

   is_ignore  
0      False  
1       True  
2      False  
3      False  
4      False  


In [286]:
# is_ignore was set by ArkhamPopularityEngine (D4: taboo_set, unknown slots,
# Chapter 2). Keep non-ignored decks for analysis.

df_decks = df_all_decks[~df_all_decks['is_ignore']].copy()
print(f"{len(df_decks)} non-ignored decks")

44571 non-ignored decks


In [287]:
# Spec Y1/Y2: user_weight and Cycle.weight

user_weight_by_id = pop_engine.assign_user_weights(prepared_decks)
df_decks['user_weight'] = df_decks['id'].map(user_weight_by_id)

cycle_weight_by_cycle = pop_engine.assign_cycle_weights(
  prepared_decks,
  user_weight_by_id,
)
df_decks['cycle_weight'] = df_decks['cycle'].map(cycle_weight_by_cycle)

df_decks['deck_weight'] = df_decks['user_weight'] * df_decks['cycle_weight']

print(f"Total user_weight: {df_decks['user_weight'].sum():.1f}")
print(f"Total deck_weight: {df_decks['deck_weight'].sum():.4f}")
print('Cycle weights:', {k: round(v, 6) for k, v in sorted(cycle_weight_by_cycle.items())})

# Sanity check (spec Y2): deck counts and sum_user_weight per cycle
cycle_sanity = (
  df_decks[df_decks['cycle'].notna()]
  .groupby('cycle', as_index=False)
  .agg(deck_count=('id', 'count'), sum_user_weight=('user_weight', 'sum'))
  .sort_values('cycle')
)
cycle_sanity['sum_user_weight'] = cycle_sanity['sum_user_weight'].round(1)
cycle_sanity['cycle_weight'] = (1.0 / cycle_sanity['sum_user_weight']).round(6)
print('\nDecks and sum_user_weight by cycle (non-ignored, cycle defined):')
display(cycle_sanity)

none_decks = df_decks[df_decks['cycle'].isna()]
if len(none_decks):
  print(
    f"cycle=None: {len(none_decks)} decks, "
    f"sum_user_weight={none_decks['user_weight'].sum():.1f}"
  )

Total user_weight: 21277.5
Total deck_weight: 10.2554
Cycle weights: {1: 0.000425, 2: 0.000425, 3: 0.000425, 4: 0.000425, 5: 0.000425, 6: 0.000443, 7: 0.000443, 8: 0.000443, 9: 0.000443, 10: 0.000523, 11: 0.000591, 12: 0.00109}

Decks and sum_user_weight by cycle (non-ignored, cycle defined):


,cycle,deck_count,sum_user_weight,cycle_weight
0,1.0,1883,1057.4,0.000946
1,2.0,4549,2089.5,0.000479
2,3.0,4234,1982.5,0.000504
3,4.0,4054,1827.7,0.000547
4,5.0,5551,2353.4,0.000425
5,6.0,3367,1530.6,0.000653
6,7.0,4445,2126.0,0.000470
7,8.0,3185,1530.9,0.000653
8,9.0,4504,2256.2,0.000443
9,10.0,3796,1912.0,0.000523


cycle=None: 1 decks, sum_user_weight=1.0


In [288]:
# deck_weight is the spec popularity weight (user_weight * Cycle.weight).
# Keep group_score as an alias for downstream cells until they are migrated.
df_decks['group_score'] = df_decks['deck_weight']
print(f"Total group_score (deck_weight): {df_decks['group_score'].sum():.4f}")

Total group_score (deck_weight): 10.2554


In [289]:
# Helper used throughout: adjust scores for uneven pack / cycle availability


def group_and_score(df: pd.DataFrame,
                    groupby_cols: list[str],
                    score_col: str,
                    pack_index_col: str):
  """Aggregate scores per group, normalized by pack availability.

  This is a direct analogue of the function in prepare_data.ipynb,
  specialized for Arkham where `pack_index_col` is publication cycle.
  """
  # For each (group, pack), total score
  df_group_pack = (
    df.groupby(groupby_cols + [pack_index_col])[score_col]
      .sum()
      .reset_index()
  )

  # For each pack, average score per group
  df_pack_map = (
    df_group_pack.groupby(pack_index_col)[score_col]
      .mean()
      .to_dict()
  )

  # Attach pack-wise average expectation
  df_group_pack['pack_avg'] = df_group_pack[pack_index_col].map(df_pack_map)

  # For each group, total score and total expected score
  df_group = (
    df_group_pack.groupby(groupby_cols)[[score_col, 'pack_avg']]
      .sum()
      .reset_index()
  )

  df_group['score'] = df_group[score_col] / df_group['pack_avg']
  return df_group.sort_values('score', ascending=False)

In [290]:
# Investigator popularity (spec I1–I5): compare investigators within each inv_cycle.
# Pooling all inv_cycles in one table overweights late-cycle investigators because
# Decklist.cycle >= 12 decks are weighted heavily under g(C) = C.

inv_name_map = (
  df_decks[['canonical_front', 'canonical_back', 'investigator_name']]
  .drop_duplicates(['canonical_front', 'canonical_back'])
  .set_index(['canonical_front', 'canonical_back'])['investigator_name']
  .to_dict()
)

inv_pop_rows = pop_engine.investigator_popularity_by_cycle(prepared_decks)
investigator_popularity = pd.DataFrame(inv_pop_rows)
investigator_popularity['investigator_name'] = investigator_popularity.apply(
  lambda row: inv_name_map.get(
    (row['canonical_front'], row['canonical_back']),
    row['canonical_front'],
  ),
  axis=1,
)
investigator_popularity = investigator_popularity.rename(columns={
  'inv_cycle': 'Cycle',
  'popularity': 'Popularity',
})

display_cols = [
  'investigator_name', 'canonical_front', 'canonical_back',
  'Cycle', 'Popularity', 'investigator_weight', 'pool_weight',
]

pd.set_option('display.max_rows', None)
for inv_cycle in sorted(investigator_popularity['Cycle'].dropna().unique()):
  print(f'=== inv_cycle = {int(inv_cycle)} (pool: Decklist.cycle >= {int(inv_cycle)}) ===')
  subset = (
    investigator_popularity[investigator_popularity['Cycle'] == inv_cycle]
    .sort_values('Popularity', ascending=False)
  )
  display(subset[display_cols])

=== inv_cycle = 1 (pool: Decklist.cycle >= 1) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,investigator_weight,pool_weight
11,Roland Banks,01001,01001,1,0.051136,0.514172,10.055071
1,Daisy Walker,01002,01002,1,0.044181,0.444242,10.055071
4,Agnes Baker,01004,01004,1,0.034920,0.351127,10.055071
7,"""Skids"" O'Toole",01003,01003,1,0.025694,0.258354,10.055071
2,Wendy Adams,01005,01005,1,0.018866,0.189698,10.055071
0,Roland Banks,01001,90024,1,0.000539,0.005415,10.055071
6,Daisy Walker,01002,90001,1,0.000450,0.004524,10.055071
10,"""Skids"" O'Toole",01003,90008,1,0.000443,0.004452,10.055071
9,Agnes Baker,01004,90017,1,0.000201,0.002018,10.055071
12,Wendy Adams,01005,90037,1,0.000155,0.001557,10.055071


=== inv_cycle = 2 (pool: Decklist.cycle >= 2) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,investigator_weight,pool_weight
14,Zoey Samaras,02001,02001,2,0.055242,0.530655,9.606063
15,Jenny Barnes,02003,02003,2,0.040735,0.391300,9.606063
21,Rex Murphy,02002,02002,2,0.028525,0.274017,9.606063
17,Jim Culver,02004,02004,2,0.021475,0.206291,9.606063
19,"""Ashcan"" Pete",02005,02005,2,0.021287,0.204488,9.606063
20,Zoey Samaras,02001,90059,2,0.000113,0.001090,9.606063
13,Jim Culver,02004,90049,2,0.000108,0.001034,9.606063
16,Jenny Barnes,02003,90084,2,0.000061,0.000591,9.606063
18,"""Ashcan"" Pete",02005,90046,2,0.000054,0.000523,9.606063


=== inv_cycle = 3 (pool: Decklist.cycle >= 3) ===


...

,investigator_name,canonical_front,canonical_back,Cycle,Popularity,investigator_weight,pool_weight
62,Charlie Kane,09018,09018,10,0.035264,0.102360,2.902695
61,Kymani Jones,09008,09008,10,0.020685,0.060042,2.902695
64,Vincent Lee,09004,09004,10,0.019834,0.057573,2.902695
63,Carson Sinclair,09001,09001,10,0.017182,0.049874,2.902695
60,Darrell Simmons,09015,09015,10,0.009047,0.026259,2.902695
59,Amina Zidane,09011,09011,10,0.008580,0.024905,2.902695


=== inv_cycle = 11 (pool: Decklist.cycle >= 11) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,investigator_weight,pool_weight
66,Alessandra Zorzi,10009,10009,11,0.032690,0.063151,1.931798
68,Wilson Richards,10001,10001,11,0.023961,0.046289,1.931798
65,Kate Winthrop,10004,10004,11,0.019678,0.038014,1.931798
67,Hank Samson,10015,10015,11,0.016812,0.032478,1.931798
69,Kōhaku Narukami,10012,10012,11,0.015777,0.030477,1.931798


=== inv_cycle = 12 (pool: Decklist.cycle >= 12) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,investigator_weight,pool_weight
75,George Barnaby,11017,11017,12,0.064349,0.061891,0.961795
76,Michael McGlen,11011,11011,12,0.062791,0.060392,0.961795
74,Marion Tavares,11001,11001,12,0.055893,0.053757,0.961795
72,Lucius Galloway,11004,11004,12,0.043659,0.041991,0.961795
71,Gloria Goldberg,11014,11014,12,0.043164,0.041515,0.961795
70,Agatha Crane,11008,11008,12,0.027055,0.026021,0.961795
73,Agatha Crane,11007,11007,12,0.019786,0.019030,0.961795


In [291]:
# Card popularity within a single investigator (spec P1–P5 + bias compensation)

DISPLAY_COLS = [
  'canonical_id', 'card_index', 'option_index', 'name',
  'cycle', 'xp', 'slot', 'P3', 'P4', 'Popularity',
]


def extract_investigator_cards(canonical_front: str,
                               canonical_back: str,
                               min_popularity: float = 0.0) -> pd.DataFrame:
  """Return spec P1–P5 popularity for a (canonical_front, canonical_back) tuple."""
  rows = pop_engine.popularity_for_investigator(
    prepared_decks,
    canonical_front,
    canonical_back,
  )
  df = pd.DataFrame(rows)
  if df.empty:
    return df
  df['cycle'] = df['canonical_id'].map(
    lambda cid: pop_engine.canonical_cards[cid].cycle
    if cid in pop_engine.canonical_cards else None
  )
  df = df[df['p5_popularity'] >= min_popularity]
  return df.rename(columns={
    'p3_opportunity_weight': 'P3',
    'p4_choice_weight': 'P4',
    'p5_popularity': 'Popularity',
  }).sort_values('Popularity', ascending=False)


def show_investigator_card_popularity(canonical_front: str,
                                      canonical_back: str,
                                      *,
                                      min_popularity: float = 0.0,
                                      head: int = 40) -> None:
  """Show separate 0 XP and 1+ XP popularity tables with CanonicalCard.cycle."""
  df = extract_investigator_cards(
    canonical_front, canonical_back, min_popularity=min_popularity,
  )
  inv_name = (
    df_decks.loc[
      (df_decks['canonical_front'] == canonical_front)
      & (df_decks['canonical_back'] == canonical_back),
      'investigator_name',
    ].iloc[0]
    if len(df_decks) else canonical_front
  )
  print(f'{inv_name} ({canonical_front} / {canonical_back})')

  zero_xp = df[df['xp'].fillna(0) == 0].copy()
  plus_xp = df[df['xp'].fillna(0) > 0].copy()

  # Format display columns: ints where applicable; blank (not 'None') for N/A text fields
  for table in [zero_xp, plus_xp]:
    if 'card_index' in table.columns:
      table['card_index'] = table['card_index'].fillna(0).astype(int)
    if 'cycle' in table.columns:
      table['cycle'] = table['cycle'].fillna(0).astype(int)
    if 'xp' in table.columns:
      table['xp'] = table['xp'].fillna(0).astype(int)
    if 'option_index' in table.columns:
      table['option_index'] = table['option_index'].fillna('')
    if 'slot' in table.columns:
      table['slot'] = table['slot'].fillna('')

  print(f'\n--- 0 XP cards ({len(zero_xp)} options) ---')
  display(zero_xp[DISPLAY_COLS].head(head))

  print(f'\n--- 1+ XP cards ({len(plus_xp)} options) ---')
  display(plus_xp[DISPLAY_COLS].head(head))


# Example: Daisy Walker (03002)
show_investigator_card_popularity('03002', '03002')

Minh Thi Phan (03002 / 03002)

--- 0 XP cards (444 options) ---


,canonical_id,card_index,option_index,name,cycle,xp,slot,P3,P4,Popularity
0,03010,1,,Analytical Mind,3,0,,0.598680,0.598680,1.000000
1,03011,1,,The King in Yellow,3,0,Hand,0.598680,0.598680,1.000000
2,01039,1,,Deduction,1,0,,0.855430,0.754922,0.881970
3,01039,2,,Deduction,1,0,,0.855430,0.724139,0.844811
4,01000,1,,Random Basic Weakness,1,0,,0.855430,0.661550,0.774807
5,01090,1,,Perception,1,0,,0.855430,0.621489,0.727791
6,01090,2,,Perception,1,0,,0.855430,0.556182,0.650652
7,02227,1,,Inquiring Mind,2,0,,0.678354,0.417525,0.624700
8,03231,1,,Eureka!,3,0,,0.598680,0.332933,0.573504
9,01093,1,,Unexpected Courage,1,0,,0.855430,0.477261,0.569614



--- 1+ XP cards (260 options) ---


,canonical_id,card_index,option_index,name,cycle,xp,slot,P3,P4,Popularity
18,60228,1,,Perception,7,2,,0.385810,0.133588,0.386022
23,05194,1,,Grisly Totem,5,3,Accessory,0.380989,0.113754,0.353409
24,60228,2,,Perception,7,2,,0.385810,0.118240,0.351804
29,11049,1,,Antikythera,12,5,Accessory,0.392499,0.108108,0.275435
32,02150,1,,Deduction,2,2,,0.335828,0.070061,0.257606
35,07196,1,,Unrelenting,8,1,,0.421397,0.075540,0.244162
40,01040,1,,Magnifying Glass,1,1,Hand,0.405371,0.096488,0.233203
41,02150,2,,Deduction,2,2,,0.335828,0.062195,0.232256
43,10118,1,,Persistence,11,1,,0.464913,0.107314,0.231335
45,06204,1,,Sharp Vision,6,1,,0.374069,0.058822,0.228170


In [292]:
# Number of assets in each slot (spec: weighted avg per asset slot type)

from collections import defaultdict

from arkham_popularity import STANDARD_ASSET_SLOT_TYPES, asset_slot_counts

SLOT_USAGE_COLS = ['slot_type', 'weighted_avg', 'deck_count', 'total_weight']


def slot_usage_for_investigator(canonical_front: str,
                                canonical_back: str) -> pd.DataFrame:
  """Weighted average asset copies per asset slot type for one investigator tuple."""
  inv_decks = [
    deck for deck in prepared_decks
    if deck.canonical_front == canonical_front
    and deck.canonical_back == canonical_back
    and not deck.is_ignore
    and deck.cycle is not None
  ]
  user_weights = pop_engine.assign_user_weights(prepared_decks)
  cycle_weights = pop_engine.assign_cycle_weights(inv_decks, user_weights)

  slot_totals: dict[str, float] = defaultdict(float)
  total_weight = 0.0
  for deck in inv_decks:
    weight = pop_engine.deck_weight(deck, user_weights, cycle_weights)
    if not weight:
      continue
    total_weight += weight
    for canonical_id, count in deck.slots.items():
      card = pop_engine.cards.get(canonical_id)
      if card is None or card.get('type_code') != 'asset':
        continue
      info = pop_engine.canonical_cards.get(canonical_id)
      if info is None:
        continue
      # Sled Dog (08127): half an Ally slot per copy, not a full Ally each.
      per_slot = asset_slot_counts(
        canonical_id,
        info.slot,
        info.real_slot,
        count,
        name=info.name,
      )
      for slot_type, slot_copies in per_slot.items():
        slot_totals[slot_type] += weight * slot_copies

  if not total_weight:
    return pd.DataFrame()
  rows = [
    {
      'canonical_front': canonical_front,
      'canonical_back': canonical_back,
      'slot_type': slot_type,
      'weighted_avg': slot_totals.get(slot_type, 0.0) / total_weight,
      'deck_count': len(inv_decks),
      'total_weight': total_weight,
    }
    for slot_type in STANDARD_ASSET_SLOT_TYPES
  ]
  return pd.DataFrame(rows)


def show_slot_usage_for_investigator(canonical_front: str,
                                     canonical_back: str) -> None:
  """Display weighted average assets per asset slot type (Accessory, Ally, ...)."""
  df = slot_usage_for_investigator(canonical_front, canonical_back)
  inv_name = (
    df_decks.loc[
      (df_decks['canonical_front'] == canonical_front)
      & (df_decks['canonical_back'] == canonical_back),
      'investigator_name',
    ].iloc[0]
    if len(df_decks) else canonical_front
  )
  print(f'{inv_name} ({canonical_front} / {canonical_back})')
  if df.empty:
    print('No active decks for this investigator.')
    return
  display(
    df[SLOT_USAGE_COLS]
    .assign(weighted_avg=lambda t: t['weighted_avg'].round(3))
    .sort_values('weighted_avg', ascending=False)
    .reset_index(drop=True)
  )


# Example: Minh Thi Phan
show_slot_usage_for_investigator('09018', '09018')

Charlie Kane (09018 / 09018)


,slot_type,weighted_avg,deck_count,total_weight
0,Ally,11.048,369,2.910431
1,Hand,3.007,369,2.910431
2,Arcane,0.807,369,2.910431
3,Accessory,0.714,369,2.910431
4,Body,0.517,369,2.910431
5,Mask,0.271,369,2.910431
6,Tarot,0.037,369,2.910431
7,Head,0.000,369,2.910431


In [293]:
# Automatic decklist generation (spec: Automatic Decklist Generation)
# Re-run cell 7 first if you only edited arkham_*.py without re-running it.

from arkham_popularity import ArkhamPopularityEngine

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)

import pandas as pd


def show_generated_decklist(
    canonical_front: str,
    canonical_back: str,
    *,
    investigator_option=None,
) -> None:
  """Build and display a synthetic 0 XP deck from popularity + slot averages.

  For investigators with deck_options choices, pass investigator_option to pick
  a branch explicitly, e.g. investigator_option='guardian+survivor' for Charlie
  Kane or investigator_option={'deck_size': 40, 'faction': 'rogue'} for Mandy.
  When omitted, the most popular meta-backed choice is used.
  """
  result = pop_engine.generate_decklist(
    prepared_decks,
    canonical_front,
    canonical_back,
    investigator_option=investigator_option,
  )
  print(
    f'{result.investigator_name} ({canonical_front} / {canonical_back})'
  )
  if result.option_resolutions:
    choices = ', '.join(
      f'{resolution.kind}={resolution.choice}'
      for resolution in result.option_resolutions
    )
    print(f'Resolved options: {choices}')
  if result.skipped_reason:
    print(f'Skipped: {result.skipped_reason}')
    return

  print(
    f'Generated {result.deck_count} / {result.deck_size} player slots '
    f'({len(result.entries)} cards shown, weaknesses omitted)'
  )
  df = pd.DataFrame(result.entries)
  if df.empty:
    display(df)
    return
  for col in ('cycle', 'xp', 'count'):
    if col in df.columns:
      df[col] = df[col].fillna(0).astype(int)
  display(df[['canonical_id', 'count', 'name', 'cycle', 'slot', 'type_code']])


show_generated_decklist('04001', '04001')

Leo Anderson (04001 / 04001)
Generated 30 / 30 player slots (22 cards shown, weaknesses omitted)


,canonical_id,count,name,cycle,slot,type_code
0,01016,2,.45 Automatic,1,Hand,asset
1,01018,2,Beat Cop,1,Ally,asset
2,01020,2,Machete,1,Hand,asset
3,01021,2,Guard Dog,1,Ally,asset
4,01025,2,Vicious Blow,1,,skill
5,01088,2,Emergency Cache,1,,event
6,01091,2,Overpower,1,,skill
7,03158,2,Calling in Favors,3,,event
8,07028,2,Faustian Bargain,8,,event
9,01048,1,Leo De Luca,1,Ally,asset


In [294]:
# Generation coverage by investigator
# Re-run cell 7 first if you only edited arkham_*.py without re-running it.

from arkham_popularity import ArkhamPopularityEngine

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)

import pandas as pd

coverage = pd.DataFrame(pop_engine.list_generatable_investigators(prepared_decks))
supported = coverage[coverage['supported']]
print(f'{len(supported)} / {len(coverage)} investigators supported for generation')
display(
  coverage.sort_values(['supported', 'name'], ascending=[False, True])[
    ['name', 'canonical_front', 'supported', 'skip_reason', 'training_decks']
  ]
)

80 / 83 investigators supported for generation


,name,canonical_front,supported,skip_reason,training_decks
10,"""Ashcan"" Pete",02005,True,None,1010
2,"""Skids"" O'Toole",01003,True,None,1044
56,Agatha Crane,11007,True,None,27
57,Agatha Crane,11008,True,None,31
3,Agnes Baker,01004,True,None,1464
14,Akachi Onyele,03004,True,None,1106
51,Alessandra Zorzi,10009,True,None,153
34,Amanda Sharpe,07002,True,None,335
46,Amina Zidane,09011,True,None,69
71,André Patel,60351,True,None,0


In [295]:
# Export generated-deck popularity CSVs (one file per supported investigator)
# Re-run cell 7 first if you only edited arkham_*.py without re-running it.

from arkham_popularity import ArkhamPopularityEngine

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)

from pathlib import Path

GENERATED_DIR = Path('generated')
written = pop_engine.export_generated_decklist_csvs(
    prepared_decks, GENERATED_DIR, diagnostics=True,
)
print(f'Wrote {len(written)} files to {GENERATED_DIR.resolve()}/')
print('Example deck csv:', next(p.name for p in written if p.suffix == '.csv' and 'resolution' not in p.name))
print('Example resolution csv:', next((p.name for p in written if 'resolution' in p.name), 'none'))

Wrote 69 files to C:\Users\jgf11\Documents\GitHub\ahlcg2\generated/
Example deck csv: Roland Banks 01001.csv
Example resolution csv: Mandy Thompson 06002 resolution.csv


In [277]:
# Bulk export above writes the default popularity/meta winner per investigator.
# For explicit deck_options branches, export a variant file instead, e.g.:
from arkham_popularity import ArkhamPopularityEngine

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)

charlie_variant = pop_engine.export_generated_decklist(
    prepared_decks,
    '09018',
    '09018',
    GENERATED_DIR,
    investigator_option='guardian+survivor',
    diagnostics=True,
)
print('Charlie variant files:', [p.name for p in charlie_variant])

Charlie variant files: ['Charlie Kane 09018 guardian+survivor.csv', 'Charlie Kane 09018 guardian+survivor resolution.csv']


In [ ]:
# Bulk export above writes the default popularity/meta winner per investigator.
# For explicit deck_options branches, export a variant file instead, e.g.:
from arkham_popularity import ArkhamPopularityEngine

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)

norman_deck = pop_engine.export_generated_decklist(
    prepared_decks,
    '08004',
    '08004',
    GENERATED_DIR,
    # investigator_option='guardian+survivor',
    diagnostics=True,
)
print('Norman variant files:', [p.name for p in charlie_variant])